In [0]:
%pip install pandas-gbq
%pip install pandas-gbq google-cloud-bigquery
%restart_python

In [0]:
# Load all necessary libraries
import requests
from dotenv import load_dotenv
import os
import json
import pandas as pd
from datetime import date, timedelta, datetime

load_dotenv()
today = date.today()

# Pulling yesterday's data. The API understands start and end date as (inclusive, inclusive)
start_date = (today - timedelta(days=1)).isoformat()
end_date = start_date

# If you ever need to run a single day, this is how it's done
# start_date = date(2026, 6, 3).isoformat()
# end_date = date(2026, 6, 3).isoformat()

print(start_date, end_date)

# This dictionary stores the project ids for each project/company - adjust for the portco prompts that you want
project_id = "or_282e29e0-ca76-49db-8e85-5dcc2208366b"

# This key gives access to all projects on the account
peek_api_key = <KEY>

In [0]:
# These functions are used to extract the data from the response in try catch blocks. Sometimes these fields don't exist in which case we return None
def extract_data_single_dimension(item, dimension):
    try:
        return item[dimension]
    except KeyError:
        return None

def extract_data_multiple_dimensions(item, dimension_one, dimension_two):
    try:
        return item[dimension_one][dimension_two]
    except (KeyError, TypeError):
        return None

def extract_list_of_ids(item, dimension_one, dimension_two):
    try:
        ids = []
        for value in item[dimension_one]:
            ids.append(value[dimension_two])
        return str(ids)
    except (KeyError, TypeError):
        return None

def extract_list_of_items(item, dimension_one, dimension_two):
    try:
        items = []
        for value in item[dimension_one]:
            items.append(value[dimension_two])
        return items
    except (KeyError, TypeError):
        return None

In [0]:
content_to_prompt_id = {
    "Are there any accelerated MBA programs in Canada for working professionals? Ideally online and at my own pace. List them and compare them.": "pr_a562e259-e794-417b-8ad6-b9620dc485c5",
    "Are there any affordable MBA programs in BC? Anything under $30,000. Then compare their benefits and negatives and suggest the best one.": "pr_c2d6f855-5274-4e0c-9436-68854609e3a3",
    "Are there any affordable MBA programs in Canada? Anything under $30,000. Then compare their benefits and negatives and suggest the best one.": "pr_d45ab756-518d-493f-b59f-7ecc70140fd8",
    "Are there any good masters in education programs in BC? Anything under $30,000. Then compare their benefits and negatives and suggest the best one.": "pr_c08a01ef-fce4-4c11-be7c-0d2a7852292d",
    "Are there any good masters in education programs in Canada? Anything under $30,000. Then compare their benefits and negatives and suggest the best one.": "pr_4f0fadad-67aa-4262-b287-06c31ae84fbd",
    "Are there any good masters in education programs in Ontario? Anything under $30,000. Then compare their benefits and negatives and suggest the best one.": "pr_e27db54f-1fd5-4309-b3b3-91f262b441c0",
    "Can you describe the role of an MBA in my business career, then offer the best online MBA degrees I could take that are available Canada-wide?": "pr_8b4bf11f-6578-4061-a7c5-51f8808c5cab",
    "Can you list the key prerequisites of a Canada BBA program, make sure to mention the schools as you list them.": "pr_edeac6d2-a908-4913-836b-b91964560570",
    "Can you recommend me some online master programs for teachers in Canada?": "pr_7d6f57c5-1088-4753-912a-b309ce1ffba3",
    "Describe the benefits of an online BBA degree, then suggest and compare the best BBA degrees in BC": "pr_4c3e1370-f6b2-4573-8e42-8b81e85c1a22",
    "Explain to me the benefits of an M.Ed degree in Canada and suggest a few affordable and potentially online M.Ed degrees in Canada.": "pr_c46a7e58-2bab-4ea5-b0b8-57402a874891",
    "How can I become a therapist in Canada? I want to know the schools to go to, how long the programs are, and what my job prospects are when I'm done the program.": "pr_f0d6e689-cb81-4b02-8c6e-3535728ba316",
    "I am a full-time highschool teacher and I want to become a principal at my school after the current principal leaves in a few years. Where can I get a masters degree to help me get this job in the meantime? I live in Canada": "pr_c3744b60-8afa-4a20-a06e-8abdc0c4e735",
    "I have $50,000 to spend on an education to earn a doctorate in counselling so I can advance my therapist career. This would have to be supplemented by government student loans. Where should I spend it?": "pr_9deec704-e513-41a7-9fae-9e5245e6a976",
    "I just landed in Canada as an immigrant and want to get a business degree, but I need to work during the day. Are there any online programs that would be good for me?": "pr_92aff647-b188-4636-a172-b0cf5a42570a",
    "I want to become a therapist, but don't have a lot of money or time. How can I earn an accredited graduate degree at my own pace online. I live in Ontario": "pr_35fefd12-aafd-4bd0-a1b2-94e0d7ae167c",
    "I'm a filmmaker with a college diploma. Are there any degrees for artists in BC wanting to get into the business side of the industry? List and compare the options.": "pr_a4a996cf-0c2a-4463-8c34-9d7db201a48d",
    "I'm a filmmaker with a college diploma. Are there any degrees for artists in Ontario wanting to get into the business side of the industry? List and compare the options.": "pr_7629a98b-8971-4111-9347-9e2cdd6712da",
    "Is an online BBA worth it? List a few key programs worth exploring as I consider my next steps.": "pr_801f980b-620d-45bf-9187-1565208b6eac",
    "What are my financial aid options for counselling psychology masters students? I'm in Ontario, but would also be helpful to know Canada wide.": "pr_1b12a132-430a-4a03-9aae-5605c9d0113c",
    "What is the ROI on a Canadian M.Ed for careers in management? And are there any high ROI M.Ed programs I should take a look at based on your research?": "pr_f1e1e793-f134-4fe9-9aa1-6e68494975a5",
    "What is the ROI on a Canadian MBA for careers in management? And are there any high ROI MBA programs I should take a look at based on your research?": "pr_35e02c95-aed8-4bc1-b098-bd786a213d24",
    "What is the difference between a masters of education in curriculum and pedagogy and a masters in educational leadership? Which one is better? And where can I earn one in Canada?": "pr_f990fe8b-9387-4793-982f-0f2dbcbd367b",
    "What is the difference between an Executive MBA vs Online MBA in Canada? Can you provide a few examples of each?": "pr_df42df20-3a9e-44bd-9f29-701542d7fd10",
}

In [0]:
import requests

rows = []
for content, prompt_id in content_to_prompt_id.items():

    url = "https://api.peec.ai/customer/v1/queries/search"

    payload = {
        "project_id": project_id,
        "limit": 1000,
        "offset": 0,
        "start_date": "2026-06-08",
        "end_date": "2026-06-14",
        "filters": [
            {
                "field": "prompt_id",
                "operator": "in",
                "values": [prompt_id]
            }
        ]
    }
    headers = {
        "X-API-Key": peek_api_key,
        "Content-Type": "application/json"
    }

    response = requests.post(url, json=payload, headers=headers)
    response = json.loads(response.text)["data"]
    print(json.dumps(response, indent=4))


    for item in response:
      row = {
          "prompt": content,
          "promt_id": prompt_id,
          "model_channel": extract_data_multiple_dimensions(item, "model_channel", "id"),
          "date": extract_data_single_dimension(item, "date"),
          "query": extract_data_multiple_dimensions(item, "query", "text"),
      }
      rows.append(row)


df = pd.DataFrame(rows)
display(df)

In [0]:
import requests

rows = []

url = "https://api.peec.ai/customer/v1/queries/search"

payload = {
    "project_id": project_id,
    "limit": 1000,
    "offset": 0,
    "start_date": "2026-06-08",
    "end_date": "2026-06-14",
    "filters": [
        {
            "field": "chat_id",
            "operator": "in",
            "values": ["ch_e165dd4a-2022-5522-a8d5-142600bd0296"]
        }
    ]
}
headers = {
    "X-API-Key": peek_api_key,
    "Content-Type": "application/json"
}

response = requests.post(url, json=payload, headers=headers)
response = json.loads(response.text)["data"]
print(json.dumps(response, indent=4))


for item in response:
    row = {
        "prompt": content,
        "promt_id": prompt_id,
        "model_channel": extract_data_multiple_dimensions(item, "model_channel", "id"),
        "date": extract_data_single_dimension(item, "date"),
        "query": extract_data_multiple_dimensions(item, "query", "text"),
    }
    rows.append(row)


df = pd.DataFrame(rows)
display(df)